In [ ]:
import subprocess, sys
def _pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import skfolio  # noqa: F401
except ImportError:
    _pip_install("skfolio")
    import skfolio  # noqa: F401

print(f"skfolio version: {skfolio.__version__}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio

try:
    pio.renderers.default = "colab"
except Exception:
    pio.renderers.default = "notebook"

from sklearn import set_config
from sklearn.model_selection import (
    GridSearchCV, KFold, train_test_split,
)
from sklearn.pipeline import Pipeline

from skfolio import RatioMeasure, RiskMeasure, Population
from skfolio.datasets import load_factors_dataset, load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from skfolio.model_selection import WalkForward, cross_val_predict

from skfolio.optimization import (
    EqualWeighted, InverseVolatility, Random,
    MeanRisk, ObjectiveFunction,
    RiskBudgeting,
    HierarchicalRiskParity,
    NestedClustersOptimization,
)
from skfolio.moments import (
    EmpiricalCovariance, LedoitWolf, DenoiseCovariance, GerberCovariance,
    EmpiricalMu, EWMu, ShrunkMu,
)
from skfolio.prior import EmpiricalPrior, BlackLitterman, FactorModel
from skfolio.pre_selection import SelectKExtremes

set_config(transform_output="pandas")

prices = load_sp500_dataset()
print("Prices shape:", prices.shape)
print(prices.tail(3))

X = prices_to_returns(prices)
print("\nReturns shape:", X.shape)

X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)
print(f"Train: {X_train.index.min().date()} → {X_train.index.max().date()}  ({len(X_train)} days)")
print(f"Test : {X_test.index.min().date()} → {X_test.index.max().date()}  ({len(X_test)} days)")

In [ ]:
benchmarks = {
    "1/N (EqualWeighted)": EqualWeighted(),
    "Inverse-Volatility":  InverseVolatility(),
    "Random (Dirichlet)":  Random(),
}

baseline_population = Population([])
for name, mdl in benchmarks.items():
    mdl.fit(X_train)
    ptf = mdl.predict(X_test)
    ptf.name = name
    baseline_population.append(ptf)
    print(f"{name:25s}  Sharpe={ptf.annualized_sharpe_ratio:.3f}  "
          f"AnnRet={ptf.annualized_mean:.3%}  AnnVol={ptf.annualized_standard_deviation:.3%}")

min_var = MeanRisk(risk_measure=RiskMeasure.VARIANCE)
min_var.fit(X_train)
print("\nMin-Variance weights (top 5):")
print(pd.Series(min_var.weights_, index=X_train.columns).sort_values(ascending=False).head())

ptf_min_var = min_var.predict(X_test)
ptf_min_var.name = "Min Variance"

max_sharpe = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
)
max_sharpe.fit(X_train)
ptf_max_sharpe = max_sharpe.predict(X_test)
ptf_max_sharpe.name = "Max Sharpe"

ef = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    efficient_frontier_size=20,
    portfolio_params=dict(name="EF"),
)
ef.fit(X_train)
ef_population_test = ef.predict(X_test)
print(f"\nEfficient frontier produced {len(ef_population_test)} portfolios.")

fig = ef_population_test.plot_measures(
    x=RiskMeasure.ANNUALIZED_VARIANCE,
    y=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
)
fig.show()

In [ ]:
risk_measures = {
    "Min CVaR-95":         RiskMeasure.CVAR,
    "Min Semi-Variance":   RiskMeasure.SEMI_VARIANCE,
    "Min CDaR":            RiskMeasure.CDAR,
    "Min Max Drawdown":    RiskMeasure.MAX_DRAWDOWN,
}

risk_pop = Population([ptf_min_var, ptf_max_sharpe])
for name, rm in risk_measures.items():
    m = MeanRisk(risk_measure=rm)
    m.fit(X_train)
    p = m.predict(X_test)
    p.name = name
    risk_pop.append(p)

print("\nRisk-measure comparison on test set:")
_summary = risk_pop.summary()
_wanted = ["Annualized Sharpe Ratio", "Annualized Sortino Ratio",
           "CVaR at 95%", "Maximum Drawdown", "Max Drawdown"]
_have = [r for r in _wanted if r in _summary.index]
print(_summary.loc[_have].T)

rb_var  = RiskBudgeting(risk_measure=RiskMeasure.VARIANCE)
rb_cvar = RiskBudgeting(risk_measure=RiskMeasure.CVAR)
rb_var.fit(X_train);   rb_cvar.fit(X_train)
ptf_rb_var  = rb_var.predict(X_test);   ptf_rb_var.name  = "Risk Parity (Var)"
ptf_rb_cvar = rb_cvar.predict(X_test);  ptf_rb_cvar.name = "Risk Parity (CVaR)"

hrp = HierarchicalRiskParity(risk_measure=RiskMeasure.VARIANCE)
hrp.fit(X_train)
ptf_hrp = hrp.predict(X_test); ptf_hrp.name = "HRP"

nco = NestedClustersOptimization(
    inner_estimator=MeanRisk(risk_measure=RiskMeasure.CVAR),
    outer_estimator=RiskBudgeting(risk_measure=RiskMeasure.VARIANCE),
    cv=KFold(n_splits=5),
    n_jobs=-1,
)
nco.fit(X_train)
ptf_nco = nco.predict(X_test); ptf_nco.name = "Nested Clusters"

hrp.hierarchical_clustering_estimator_.plot_dendrogram().show()

In [ ]:
robust = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=EmpiricalPrior(
        mu_estimator=ShrunkMu(),
        covariance_estimator=DenoiseCovariance(),
    ),
)
robust.fit(X_train)
ptf_robust = robust.predict(X_test); ptf_robust.name = "Max Sharpe (Robust)"

gerber = MeanRisk(
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=EmpiricalPrior(covariance_estimator=GerberCovariance()),
)
gerber.fit(X_train)
ptf_gerber = gerber.predict(X_test); ptf_gerber.name = "Min Var (Gerber)"

assets = list(X_train.columns)

group_a = assets[:10]; group_b = assets[10:]
groups = pd.DataFrame(
    {a: ["GroupA" if a in group_a else "GroupB"] for a in assets},
    index=["Sector"],
)

constrained = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
    min_weights=0.0,
    max_weights=0.20,
    transaction_costs=0.0005,
    groups=groups,
    linear_constraints=[
        "GroupA <= 0.6",
        "GroupB >= 0.2",
    ],
    l2_coef=0.01,
)
constrained.fit(X_train)
ptf_constr = constrained.predict(X_test); ptf_constr.name = "Constrained MV"
print("\nConstrained portfolio weights:")
print(pd.Series(constrained.weights_, index=assets).round(4))

print("\nAvailable tickers:", list(X_train.columns))

bl_views = [
    "AAPL == 0.0008",
    "JPM - BAC == 0.0002",
]
bl = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=BlackLitterman(views=bl_views),
)
bl.fit(X_train)
ptf_bl = bl.predict(X_test); ptf_bl.name = "Black-Litterman"

In [3]:
factor_prices = load_factors_dataset()
X_full, F_full = prices_to_returns(prices, factor_prices)

X_tr, X_te, F_tr, F_te = train_test_split(
    X_full, F_full, test_size=0.33, shuffle=False
)

fm = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=FactorModel(),
)
fm.fit(X_tr, F_tr)
ptf_fm = fm.predict(X_te); ptf_fm.name = "Factor Model"
print(f"\nFactor-model Sharpe: {ptf_fm.annualized_sharpe_ratio:.3f}")

pipe = Pipeline([
    ("preselect",   SelectKExtremes(k=8, highest=True)),
    ("optimize",    MeanRisk(
        objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
        risk_measure=RiskMeasure.VARIANCE)),
])
pipe.fit(X_train)
ptf_pipe = pipe.predict(X_test); ptf_pipe.name = "Top-8 + Max Sharpe"

wf_model = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
)
mp_portfolio = cross_val_predict(
    wf_model, X,
    cv=WalkForward(train_size=252*2, test_size=63),
    n_jobs=-1,
)
mp_portfolio.name = "Walk-Forward Max Sharpe"
print(f"\nWalk-forward portfolio  Sharpe={mp_portfolio.annualized_sharpe_ratio:.3f}  "
      f"CalmarRatio={mp_portfolio.calmar_ratio:.3f}")
mp_portfolio.plot_cumulative_returns().show()

tuned = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=EmpiricalPrior(mu_estimator=EWMu(alpha=0.1)),
)
grid = GridSearchCV(
    estimator=tuned,
    cv=WalkForward(train_size=252*2, test_size=63),
    n_jobs=-1,
    param_grid={
        "l2_coef": [0.0, 0.01, 0.1],
        "prior_estimator__mu_estimator__alpha": [0.05, 0.1, 0.2, 0.5],
    },
)
grid.fit(X_train)
print("\nBest params:", grid.best_params_)
print(f"Best CV score (Sharpe): {grid.best_score_:.3f}")

ptf_tuned = grid.best_estimator_.predict(X_test); ptf_tuned.name = "Tuned Max Sharpe"

final = Population([
    *baseline_population,
    ptf_min_var, ptf_max_sharpe,
    ptf_rb_var, ptf_rb_cvar,
    ptf_hrp, ptf_nco,
    ptf_robust, ptf_gerber,
    ptf_constr, ptf_bl, ptf_fm,
    ptf_pipe, ptf_tuned,
])

_full = final.summary()
_wanted_final = [
    "Annualized Mean", "Annualized Standard Deviation",
    "Annualized Sharpe Ratio", "Annualized Sortino Ratio",
    "CVaR at 95%", "Maximum Drawdown", "Max Drawdown",
]
_have_final = [r for r in _wanted_final if r in _full.index]
summary = _full.loc[_have_final].T.sort_values(
    "Annualized Sharpe Ratio", ascending=False
)

print("\n" + "=" * 80)
print("FINAL HORSE RACE — sorted by Sharpe (out-of-sample test set)")
print("=" * 80)
print(summary.to_string())

final.plot_cumulative_returns().show()

final.plot_composition().show()

ptf_rb_var.plot_contribution(measure=RiskMeasure.VARIANCE).show()

print("\nDone. Try swapping risk measures, adding constraints, or wiring in")
print("your own returns DataFrame — every estimator follows the sklearn API.")

skfolio version: 0.20.1
Prices shape: (8313, 20)
               AAPL    AMD     BAC     BBY      CVX      GE       HD      JNJ  \
Date                                                                            
2022-12-23  131.477  64.52  32.005  79.432  174.140  63.742  314.177  174.893   
2022-12-27  129.652  63.27  32.065  79.930  176.329  64.561  314.985  174.844   
2022-12-28  125.674  62.57  32.301  78.279  173.728  63.883  311.220  174.085   

                JPM      KO      LLY      MRK     MSFT      PEP     PFE  \
Date                                                                      
2022-12-23  128.421  62.855  365.762  110.350  237.614  179.781  50.249   
2022-12-27  128.871  63.240  362.760  110.607  235.852  180.580  49.570   
2022-12-28  129.575  62.609  363.098  109.581  233.434  179.278  49.250   

                 PG     RRC      UNH      WMT      XOM  
Date                                                    
2022-12-23  149.781  26.226  527.260  142.641  106.922 


Risk-measure comparison on test set:
                  Annualized Sharpe Ratio Annualized Sortino Ratio CVaR at 95%
Min Variance                         0.91                     1.28       2.16%
Max Sharpe                           1.04                     1.47       2.54%
Min CVaR-95                          0.87                     1.21       2.17%
Min Semi-Variance                    0.92                     1.27       2.15%
Min CDaR                             0.83                     1.17       2.35%
Min Max Drawdown                     0.86                     1.19       2.23%



Constrained portfolio weights:
AAPL    0.1430
AMD     0.0655
BAC     0.0000
BBY     0.1975
CVX     0.0113
GE      0.0000
HD      0.0663
JNJ     0.0103
JPM     0.0312
KO      0.0013
LLY     0.0000
MRK     0.0000
MSFT    0.0972
PEP     0.0023
PFE     0.0197
PG      0.0060
RRC     0.1660
UNH     0.1565
WMT     0.0170
XOM     0.0090
dtype: float64

Available tickers: ['AAPL', 'AMD', 'BAC', 'BBY', 'CVX', 'GE', 'HD', 'JNJ', 'JPM', 'KO', 'LLY', 'MRK', 'MSFT', 'PEP', 'PFE', 'PG', 'RRC', 'UNH', 'WMT', 'XOM']

Factor-model Sharpe: 0.768

Walk-forward portfolio  Sharpe=0.880  CalmarRatio=0.001



Best params: {'l2_coef': 0.01, 'prior_estimator__mu_estimator__alpha': 0.05}
Best CV score (Sharpe): 0.067

FINAL HORSE RACE — sorted by Sharpe (out-of-sample test set)
                    Annualized Mean Annualized Standard Deviation Annualized Sharpe Ratio Annualized Sortino Ratio CVaR at 95%
Top-8 + Max Sharpe           20.04%                        18.33%                    1.09                     1.51       2.68%
Random (Dirichlet)           17.52%                        16.40%                    1.07                     1.48       2.39%
Black-Litterman              19.56%                        18.32%                    1.07                     1.48       2.70%
Max Sharpe                   18.43%                        17.72%                    1.04                     1.47       2.54%
1/N (EqualWeighted)          17.30%                        17.15%                    1.01                     1.40       2.51%
Risk Parity (Var)            16.40%                        16.22%   


Done. Try swapping risk measures, adding constraints, or wiring in
your own returns DataFrame — every estimator follows the sklearn API.
